In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

access_id = os.getenv("ODPS_ACCESS_ID")
access_key = os.getenv("ODPS_ACCESS_KEY")
project = os.getenv("ODPS_PROJECT")
endpoint = os.getenv("ODPS_ENDPOINT")

from odps import ODPS, options

o = ODPS(
    access_id=access_id,
    secret_access_key=access_key,
    project=project,
    endpoint=endpoint,
)

options.tunnel.use_instance_tunnel = True
options.tunnel.limit_instance_tunnel = False


In [ ]:
def q(sql):
    """Run SQL on DataWorks and return the result as a pandas DataFrame."""
    inst = o.execute_sql(sql)
    inst.wait_for_success()
    with inst.open_reader(tunnel=True) as reader:
        return reader.to_pandas()


print("Connected to project:", o.project)
print(os.getenv("ODPS_ACCESS_ID"))
print(os.getenv("ODPS_ACCESS_KEY"))

In [ ]:
lookback_dt = '2026-05-01'
train_start = '2026-07-01'
train_end = '2026-08-28'
oot_start = '2026-08-28'
oot_end = '2026-10-01'

In [ ]:
sql = f"""
WITH 
-- One row per transaction, tagged with its sample; rows outside both windows are dropped
card_transactions AS (
    SELECT
        trans_id,
        nvl(async_issue_bank, 'null') AS async_issue_bank,
        SUBSTRING(TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City'), 1, 10) AS transaction_time,
        TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') AS transaction_timestamp,
        SUBSTRING(mexico_city_time, 1, 10) AS transaction_mexico_time,
        mexico_city_time AS transaction_mexico_timestamp,
        event_id,
        user_id,
        sub_merchant_id AS merchant_id,
        async_card_id AS card_id,
        async_card_pan_uuid, 
        REPLACE(async_card_last_digits, '*', '') as async_card_last_digits,
        rules,
        ROUND(event_amount / 100, 2) AS transaction_amount,
        ROW_NUMBER() OVER (PARTITION BY user_id, trans_id ORDER BY TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') DESC) AS rn,
        CASE
        WHEN TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') >= CAST(TO_DATE('{train_start}', 'yyyy-mm-dd') AS TIMESTAMP) 
        AND TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') <= CAST(TO_DATE('{train_end}', 'yyyy-mm-dd') AS TIMESTAMP) THEN 'training'
        WHEN TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') >= CAST(TO_DATE('{oot_start}', 'yyyy-mm-dd') AS TIMESTAMP) AND 
        TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') <= CAST(TO_DATE('{oot_end}', 'yyyy-mm-dd') AS TIMESTAMP)   THEN 'oot'
        END AS sample_type
    FROM BaseHub_mx.dwd_risk_event_payment_details_di
    WHERE dt >= '{lookback_dt}'
        AND ((TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') >= CAST(TO_DATE('{train_start}', 'yyyy-mm-dd') AS TIMESTAMP)
        AND TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') <= CAST(TO_DATE('{train_end}', 'yyyy-mm-dd') AS TIMESTAMP)) OR
        (TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') >= CAST(TO_DATE('{oot_start}', 'yyyy-mm-dd') AS TIMESTAMP) AND 
        TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') <= CAST(TO_DATE('{oot_end}', 'yyyy-mm-dd') AS TIMESTAMP)))
        AND pay_option LIKE '%CARD%'
),

card_transactions_dedup AS (
    SELECT
        *
    FROM card_transactions
    WHERE rn = 1
),

-- Keeps cumulating the transactions amount
card_cumamount AS (
    SELECT
        user_id,
        trans_id,
        ROUND(event_amount / 100, 2) AS transaction_amount,
        SUM(ROUND(event_amount / 100, 2)) OVER (PARTITION BY user_id ORDER BY TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City')) AS running_total_amount,
        ROW_NUMBER() OVER (PARTITION BY user_id, trans_id ORDER BY TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City') DESC) AS rn
    FROM BaseHub_mx.dwd_risk_event_payment_details_di
    WHERE pay_option LIKE '%CARD%'
),

card_cumamount_dedup AS (
    SELECT
        *
    FROM card_cumamount
    WHERE rn = 1
),

-- Disputes by date dt, creating row number to keep the last transaction register
dispute_table AS (
    SELECT
        user_id,
        trans_id, 
        COALESCE(TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City'), TO_UTC_TIMESTAMP(payment_created_at_cdmx, 'America/Mexico_City')) AS transaction_dispute_timestamp,
        DATE_FORMAT(COALESCE(TO_UTC_TIMESTAMP(mexico_city_time, 'America/Mexico_City'), TO_UTC_TIMESTAMP(payment_created_at_cdmx, 'America/Mexico_City')), 'yyyy-MM-dd') AS transaction_dispute_time, 
        DATE_FORMAT(TO_UTC_TIMESTAMP(controversy_date_cdmx, 'America/Mexico_City'), 'yyyy-MM-dd') AS controversy_date, 
        controversy_type, 
        reason_code, 
        ROUND(payment_amount, 2) AS dispute_amount, 
        currency, 
        dt,
        ROW_NUMBER() OVER (PARTITION BY user_id, trans_id ORDER BY dt DESC) AS rn
    FROM basehub_mx.dwd_controversy_dispute_miclip_match_df
    WHERE reason_code IN ('68', '41') 
),


-- Dispute table with all transactions over different dates (dt)
dispute_cases AS (
    SELECT
        DATEDIFF(controversy_date, transaction_dispute_time) AS lagtime_dispute,
        *,
        CASE WHEN lower(controversy_type) = 'dispute' THEN 1 ELSE 0 END AS controversy_type_dispute,
        CASE WHEN lower(controversy_type) = 'chargeback' THEN 1 ELSE 0 END AS controversy_type_chargeback
    FROM dispute_table
    WHERE rn = 1
    ORDER BY user_id, trans_id, transaction_dispute_timestamp, dt DESC
),

-- Disputes cumulated transaction amount 
disputes_cumamount AS (
  SELECT
        user_id,
        trans_id,
        transaction_dispute_time,
        transaction_dispute_timestamp, 
        controversy_date, 
        controversy_type, 
        reason_code, 
        currency, 
        dt,
        dispute_amount,
        SUM(ROUND(dispute_amount, 2)) OVER (PARTITION BY user_id ORDER BY transaction_dispute_timestamp) AS running_disputes_amount
    FROM dispute_cases
    WHERE reason_code IN ('68', '41')
),

-- Blacklist - Suspicious transactions - High severity fraud (once we have the amount)
suspicious_table AS (
  SELECT
    content as user_id, 
    list_name, 
    status,
    TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City') AS blacklist_time,
    dt,
    1 AS suspicious_userid,
    ROW_NUMBER() OVER (PARTITION BY content, TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City') ORDER BY dt DESC) AS rn
FROM BaseHub_mx.ods_risk_variable_black_list
WHERE status = 1 AND (list_name IN ('Supicious_userId', 'Suspicious_userId','Suspicipus_userId')) --1,313 users
ORDER BY user_id, TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City'), dt DESC
),

suspicious_user AS (
  SELECT
    user_id, 
    list_name, 
    status,
    blacklist_time,
    dt,
    suspicious_userid
FROM suspicious_table
WHERE rn = 1 --1,313 users
ORDER BY user_id, blacklist_time, dt DESC
),

-- Blacklist - Stolen Card and Account Take Over transactions
cs_ato_table AS (
  SELECT
    content as user_id, 
    list_name, 
    status,
    TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City') AS blacklist_time, 
    dt,
    CASE WHEN list_name = 'CS_Black_Userid' AND status = 1 THEN 1 ELSE NULL END AS cs_userid, --18 users
    CASE WHEN list_name = 'ATO_Black_UserId' AND status = 1 THEN 1 ELSE NULL END AS ato_userid, --3 users
    ROW_NUMBER() OVER (PARTITION BY content, id, TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City') ORDER BY dt DESC) AS rn
FROM BaseHub_mx.ods_risk_variable_black_list
WHERE status = 1 AND (list_name = 'CS_Black_Userid' OR list_name = 'ATO_Black_UserId')
ORDER BY content, TO_UTC_TIMESTAMP(gmt_create, 'America/Mexico_City'), dt DESC
),

cs_ato_user AS (
  SELECT
    user_id, 
    list_name, 
    status,
    blacklist_time, 
    dt,
    cs_userid, 
    ato_userid
FROM cs_ato_table
WHERE rn = 1 
ORDER BY user_id, blacklist_time, dt DESC
),

-- Transaction-level detail with pivoted flags and stolencard_label
txns_disputes_blacklist AS (
    SELECT
    t.sample_type,
    t.trans_id,
    t.async_issue_bank,
    t.transaction_time,
    t.transaction_timestamp,
    t.transaction_mexico_time,
    t.transaction_mexico_timestamp,
    t.event_id,
    t.user_id,
    nvl(t.merchant_id, 'null') AS merchant_id,
    nvl(t.card_id, 'null') AS card_id,
    t.transaction_amount,
    cc.running_total_amount,
    nvl(d.reason_code, 'null') AS reason_code,
    nvl(dc.dispute_amount, 0) AS dispute_amount,
    nvl(dc.running_disputes_amount, 0) AS running_disputes_amount,
    CASE WHEN nvl(dc.running_disputes_amount, 0)/cc.running_total_amount > 0.1 THEN 1 ELSE 0 END AS dispute_by_amount_greater10p,
    nvl(d.controversy_type_dispute, 0) AS controversy_type_dispute,
    nvl(d.controversy_type_chargeback, 0) AS controversy_type_chargeback,
    CASE WHEN cs.cs_userid = 1 THEN 1 ELSE 0 END AS blacklist_stolencard_label,
    CASE WHEN cs.ato_userid = 1 THEN 1 ELSE 0 END AS blacklist_ato_label,
    CASE WHEN s.suspicious_userid = 1 AND t.transaction_amount > 2000 THEN 1 ELSE 0 END AS blacklist_suspicious_highsev_label
    FROM card_transactions_dedup t
    LEFT JOIN card_cumamount_dedup cc
    ON t.user_id = cc.user_id AND t.trans_id = cc.trans_id  
    LEFT JOIN dispute_cases d
    ON t.user_id = d.user_id AND t.trans_id = d.trans_id  
    LEFT JOIN disputes_cumamount dc
    ON t.user_id = dc.user_id AND t.trans_id = dc.trans_id  
    LEFT JOIN suspicious_user s
    ON t.user_id = s.user_id AND t.transaction_timestamp >= s.blacklist_time
    LEFT JOIN cs_ato_user cs
    ON t.user_id = cs.user_id AND t.transaction_timestamp >= cs.blacklist_time
),

disputes_amt_gt_10p AS (
    SELECT 
        user_id,
        MAX(dispute_by_amount_greater10p) AS disputes_amt_gt_10p_label
    FROM txns_disputes_blacklist
    GROUP BY user_id
),

labeled AS (
    SELECT 
        t.*,
        d.disputes_amt_gt_10p_label,
        CASE WHEN d.disputes_amt_gt_10p_label = 1 THEN 1 ELSE 0 END AS controversy_type_dispute_txngrt10p,
        CASE WHEN nvl(t.controversy_type_dispute, 0) = 1 OR nvl(t.controversy_type_chargeback, 0) = 1 OR nvl(d.disputes_amt_gt_10p_label, 0) = 1 THEN 1 ELSE 0 END AS disputes_label,
        CASE WHEN nvl(t.blacklist_stolencard_label, 0) = 1 OR nvl(t.blacklist_ato_label, 0) = 1 OR nvl(t.blacklist_suspicious_highsev_label, 0) = 1 THEN 1 ELSE 0 END AS blacklist_label,
        CASE WHEN nvl(t.controversy_type_dispute, 0) = 1 OR nvl(t.controversy_type_chargeback, 0) = 1 OR nvl(d.disputes_amt_gt_10p_label, 0) = 1 
            OR nvl(t.blacklist_stolencard_label, 0) = 1 OR nvl(t.blacklist_ato_label, 0) = 1 OR nvl(t.blacklist_suspicious_highsev_label, 0) = 1 THEN 1 ELSE 0 END AS fraud_label
    FROM txns_disputes_blacklist AS t LEFT JOIN disputes_amt_gt_10p AS d ON t.user_id = d.user_id
),

-- Splitting training and out of time users to prevent the same user to be in both samples
training_users AS (
    SELECT 
        DISTINCT user_id
    FROM labeled
    WHERE sample_type = 'training'
),

train_txns AS (
    SELECT 
        *
    FROM labeled
    WHERE sample_type = 'training'
),

oot_txns AS (
    SELECT
        *
    FROM labeled  
    WHERE sample_type = 'oot'
),

oot_clean_txns AS (
    SELECT 
        o.*
    FROM oot_txns o LEFT JOIN training_users t ON o.user_id = t.user_id 
    WHERE t.user_id IS NULL
),

labeled_exclusive_samples AS (
    SELECT * FROM train_txns
    UNION ALL
    SELECT * FROM oot_clean_txns
),

-- Final output: transaction-level detail.
-- user_id, trans_id, card_id, merchant_id (features)
 
-- Summary per sample_type: observed transactions, disputes, chargebacks, stolen card cases
summary_by_sample AS (
  SELECT
    sample_type,
    count(trans_id)                        AS observed_transactions,
    count(DISTINCT trans_id)               AS observed_unique_transactions,
    sum(controversy_type_dispute)          AS dispute_cnt,
    sum(controversy_type_chargeback)       AS chargeback_cnt,
    sum(controversy_type_dispute_txngrt10p) AS controversy_type_dispute_txngrt10p_cnt,
    sum(disputes_label)         AS disputes_cnt,
    sum(disputes_label) / count(DISTINCT trans_id) AS disputes_rate,
    sum(blacklist_stolencard_label)  AS blacklist_stolencard_cnt,
    sum(blacklist_stolencard_label) / count(DISTINCT trans_id) AS blacklist_stolencard_rate,
    sum(blacklist_ato_label)  AS blacklist_ato_cnt,
    sum(blacklist_ato_label) / count(DISTINCT trans_id) AS blacklist_ato_rate,
    sum(blacklist_suspicious_highsev_label)  AS blacklist_suspicious_highsev_cnt,
    sum(blacklist_suspicious_highsev_label) / count(DISTINCT trans_id) AS blacklist_suspicious_highsev_rate,    
    sum(blacklist_label)  AS blacklist_cnt,
    sum(blacklist_label) / count(DISTINCT trans_id) AS blacklist_rate,
    sum(fraud_label)  AS fraud_cnt,
    sum(fraud_label) / count(DISTINCT trans_id) AS fraud_rate,
    MIN(transaction_timestamp) AS min_utc_date,
    MAX(transaction_timestamp) AS max_utc_date,
    MIN(transaction_mexico_timestamp) AS min_mexico_date,
    MAX(transaction_mexico_timestamp) AS max_mexico_date
  FROM labeled_exclusive_samples
  GROUP BY sample_type
)

-- To get the summary instead, comment out the line above and use:
SELECT * FROM summary_by_sample;
"""

instance = o.execute_sql(sql)
instance.wait_for_success()

In [ ]:
import pandas as pd

with instance.open_reader(tunnel=True) as reader:
    summary = reader.to_pandas()

summary

In [ ]:
sql = f"""
    SELECT * FROM BaseHub_mx_dev.stolencard_dataset_all_tables;
"""

instance = o.execute_sql(sql)
instance.wait_for_success()

In [ ]:
import pandas as pd

with instance.open_reader(tunnel=True) as reader:
    data = reader.to_pandas()

data

In [ ]:
groupped = data.groupby('sample_type')['trans_id'].nunique().reset_index()
groupped

In [ ]:
data['transaction_time'] = pd.to_datetime(data['transaction_time'])

data_weekly = data.resample('W-SUN', on='transaction_time').agg({'trans_id': 'count', 'fraud_label': 'sum'}).reset_index()
data_weekly['Week_Ending_Label'] = data_weekly['transaction_time'].dt.strftime('%Y-%m-%d')

# Calculate the actual aggregate Rate (e.g., Revenue per Unit)
data_weekly['fraud_rate'] = data_weekly['fraud_label'] / data_weekly['trans_id']

data_weekly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 3. Plot the Dual Axis Figure ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Primary Axis: Volume as a Barplot
bars = sns.barplot(data=data_weekly, x='Week_Ending_Label', y='trans_id', ax=ax1, color='skyblue', alpha=0.8)
ax1.set_ylabel('# of transactions', color='skyblue', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='skyblue')
ax1.set_xlabel('week ending', fontsize=12)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
ax1.bar_label(bars.containers[0], fmt='%d', label_type='edge', padding=4, color='navy', fontweight='bold')


# Secondary Axis: Rate as a Lineplot
ax2 = ax1.twinx()
# Match line coordinates to the categorical positions of the barplot via ax1.get_xticks()
sns.lineplot(x=ax1.get_xticks(), y=data_weekly['fraud_rate'], ax=ax2, color='crimson', marker='o', linewidth=2.5)
ax2.set_ylabel('stolen card fraud rate (%)', color='crimson', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='crimson')

# Add Labels to Line Points by looping through coordinates
for x, y in zip(ax1.get_xticks(), data_weekly['fraud_rate']):
    ax2.annotate(
        f"{y*100:.2f}%",             # Label formatting string
        (x, y),                  # Target data point coordinate
        textcoords="offset points", 
        xytext=(0, 10),          # Push the text 10 points vertically above the marker
        ha='center',             # Horizontally center text
        color='crimson', 
        fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="crimson", alpha=0.8, lw=0.5) # Optional background bubble for high-contrast visibility
    )

# Cleanup & Presentation
ax2.grid(False) 
plt.title('weekly transactions vs. stolen card fraud rate', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()

plt.show()

In [ ]:
intime = data.loc[data['sample_type'] == 'training']

intime_weekly = intime.resample('W-SUN', on='transaction_time').agg({'trans_id': 'count', 'fraud_label': 'sum'}).reset_index()
intime_weekly['Week_Ending_Label'] = intime_weekly['transaction_time'].dt.strftime('%Y-%m-%d')

# Calculate the actual aggregate Rate (e.g., Revenue per Unit)
intime_weekly['fraud_rate'] = intime_weekly['fraud_label'] / intime_weekly['trans_id']

intime_weekly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 3. Plot the Dual Axis Figure ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Primary Axis: Volume as a Barplot
bars = sns.barplot(data=intime_weekly, x='Week_Ending_Label', y='trans_id', ax=ax1, color='skyblue', alpha=0.8)
ax1.set_ylabel('# of transactions', color='skyblue', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='skyblue')
ax1.set_xlabel('week ending', fontsize=12)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
ax1.bar_label(bars.containers[0], fmt='%d', label_type='edge', padding=4, color='navy', fontweight='bold')


# Secondary Axis: Rate as a Lineplot
ax2 = ax1.twinx()
# Match line coordinates to the categorical positions of the barplot via ax1.get_xticks()
sns.lineplot(x=ax1.get_xticks(), y=intime_weekly['fraud_rate'], ax=ax2, color='crimson', marker='o', linewidth=2.5)
ax2.set_ylabel('stolen card fraud rate (%)', color='crimson', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='crimson')

# Add Labels to Line Points by looping through coordinates
for x, y in zip(ax1.get_xticks(), intime_weekly['fraud_rate']):
    ax2.annotate(
        f"{y*100:.2f}%",             # Label formatting string
        (x, y),                  # Target data point coordinate
        textcoords="offset points", 
        xytext=(0, 10),          # Push the text 10 points vertically above the marker
        ha='center',             # Horizontally center text
        color='crimson', 
        fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="crimson", alpha=0.8, lw=0.5) # Optional background bubble for high-contrast visibility
    )

# Cleanup & Presentation
ax2.grid(False) 
plt.title('In time sample - weekly transactions vs. stolen card fraud rate', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()

plt.show()

In [ ]:
outoftime = data.loc[data['sample_type'] == 'oot']

outoftime_weekly = outoftime.resample('W-SUN', on='transaction_time').agg({'trans_id': 'count', 'fraud_label': 'sum'}).reset_index()
outoftime_weekly['Week_Ending_Label'] = outoftime_weekly['transaction_time'].dt.strftime('%Y-%m-%d')

# Calculate the actual aggregate Rate (e.g., Revenue per Unit)
outoftime_weekly['fraud_rate'] = outoftime_weekly['fraud_label'] / outoftime_weekly['trans_id']

outoftime_weekly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 3. Plot the Dual Axis Figure ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Primary Axis: Volume as a Barplot
bars = sns.barplot(data=outoftime_weekly, x='Week_Ending_Label', y='trans_id', ax=ax1, color='skyblue', alpha=0.8)
ax1.set_ylabel('# of transactions', color='skyblue', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='skyblue')
ax1.set_xlabel('week ending', fontsize=12)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
ax1.bar_label(bars.containers[0], fmt='%d', label_type='edge', padding=4, color='navy', fontweight='bold')


# Secondary Axis: Rate as a Lineplot
ax2 = ax1.twinx()
# Match line coordinates to the categorical positions of the barplot via ax1.get_xticks()
sns.lineplot(x=ax1.get_xticks(), y=outoftime_weekly['fraud_rate'], ax=ax2, color='crimson', marker='o', linewidth=2.5)
ax2.set_ylabel('stolen card fraud rate (%)', color='crimson', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='crimson')

# Add Labels to Line Points by looping through coordinates
for x, y in zip(ax1.get_xticks(), outoftime_weekly['fraud_rate']):
    ax2.annotate(
        f"{y*100:.2f}%",             # Label formatting string
        (x, y),                  # Target data point coordinate
        textcoords="offset points", 
        xytext=(0, 10),          # Push the text 10 points vertically above the marker
        ha='center',             # Horizontally center text
        color='crimson', 
        fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="crimson", alpha=0.8, lw=0.5) # Optional background bubble for high-contrast visibility
    )

# Cleanup & Presentation
ax2.grid(False) 
plt.title('Out of time sample - weekly transactions vs. stolen card fraud rate', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()

plt.show()